# IFC2x3 Duplex Architecture - COBie Data Transformation

This notebook processes IFC model JSON to add COBie values from an Excel reference table.
- **Input**: `JSON Whole Model/Ifc2x3_Duplex_Architecture.json` and COBie Excel lookup
- **Output**: `JSON_Edit/Ifc2x3_Duplex_Architecture.json` with COBie values added
- **Processed Items**: Walls, Windows, Doors


In [1]:
from pathlib import Path
from shutil import copy2
import json
import pandas as pd
from IPython.display import display

# ============================================================
# SETUP & PATHS
# ============================================================

workspace_root = (Path.cwd() / '..').resolve()
source_json_path = workspace_root / 'JSON Whole Model' / 'Ifc2x3_Duplex_Architecture.json'
excel_path = workspace_root / 'COBie' / 'Uniclass2015_EF_v1_16.xlsx'
json_edit_dir = workspace_root / 'JSON_Edit'

# Verify files exist
assert source_json_path.exists(), f'JSON not found: {source_json_path}'
assert excel_path.exists(), f'Excel not found: {excel_path}'

json_edit_dir.mkdir(parents=True, exist_ok=True)
working_json_path = json_edit_dir / source_json_path.name

# Backup source if working copy doesn't exist
if not working_json_path.exists():
    copy2(source_json_path, working_json_path)

print(f'Source JSON: {source_json_path}')
print(f'Working JSON: {working_json_path}')
print(f'Excel Reference: {excel_path}')


Source JSON: C:\Git\APS-IFC\JSON Whole Model\Ifc2x3_Duplex_Architecture.json
Working JSON: C:\Git\APS-IFC\JSON_Edit\Ifc2x3_Duplex_Architecture.json
Excel Reference: C:\Git\APS-IFC\COBie\Uniclass2015_EF_v1_16.xlsx


In [2]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_prop_value(properties, category, display_name):
    """Get a property value from items's Properties list."""
    if not isinstance(properties, list):
        return None
    for prop in properties:
        if not isinstance(prop, dict):
            continue
        if (str(prop.get('category', '')).strip().upper() == category and 
            str(prop.get('displayName', '')).strip().upper() == display_name):
            return str(prop.get('value', '')).strip()
    return None


def is_type_match(item, target_type):
    """Check if item.Properties has Item/Type matching target_type."""
    properties = item.get('Properties', []) if isinstance(item, dict) else []
    if not isinstance(properties, list):
        return False
    
    for prop in properties:
        if not isinstance(prop, dict):
            continue
        category = str(prop.get('category', '')).strip().lower()
        display_name = str(prop.get('displayName', '')).strip().lower()
        value = str(prop.get('value', '')).strip().upper()
        
        if category == 'item' and display_name == 'type' and value == target_type:
            return True
    return False


def extract_items_by_type(data, target_type):
    """Extract all items matching a specific IFC Type from source data."""
    rows = []
    for item in data:
        if not isinstance(item, dict):
            continue
        if is_type_match(item, target_type):
            rows.append({
                'GUID': str(item.get('ExternalId', '')).strip(),
                'Name': item.get('Name', ''),
                'DbId': item.get('DbId'),
                'Item.Type': target_type,
            })
    
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(by=['Name', 'GUID'], kind='stable').reset_index(drop=True)
        df.insert(0, 'Count', df.index + 1)
    
    return df


def apply_cobie_value(target_item, cobie_value):
    """Apply COBie value to an item, returning (was_updated, before_value, after_value)."""
    properties = target_item.get('Properties')
    if not isinstance(properties, list):
        properties = []
        target_item['Properties'] = properties
    
    existing_cobie_prop = None
    for prop in properties:
        if not isinstance(prop, dict):
            continue
        if (str(prop.get('category', '')).strip() == 'IFC' and 
            str(prop.get('displayName', '')).strip() == 'COBie'):
            existing_cobie_prop = prop
            break
    
    before_value = None
    if existing_cobie_prop is not None:
        before_value = str(existing_cobie_prop.get('value', '')).strip() or '(empty)'
        if before_value != cobie_value:
            existing_cobie_prop['value'] = cobie_value
            return True, before_value, cobie_value
        else:
            return False, before_value, cobie_value
    else:
        properties.append({
            'category': 'IFC',
            'displayName': 'COBie',
            'value': cobie_value,
        })
        return True, '(none)', cobie_value


In [3]:
# ============================================================
# LOAD DATA & SEARCH EXCEL FOR COBIE MAPPINGS
# ============================================================

# Load source and working data
with source_json_path.open('r', encoding='utf-8') as f:
    source_data = json.load(f)

with working_json_path.open('r', encoding='utf-8') as f:
    working_data = json.load(f)

# Load Excel COBie reference table
ef_df = pd.read_excel(excel_path, sheet_name='EF', header=2)

# Define what to process: IFC type -> Excel title search term
items_to_process = [
    {'ifc_type': 'IFCWALL', 'excel_search': 'wall', 'display_name': 'Walls'},
    {'ifc_type': 'IFCDOOR', 'excel_search': 'door', 'display_name': 'Doors'},
    {'ifc_type': 'IFCWINDOW', 'excel_search': 'window', 'display_name': 'Windows'},
]

# Search Excel and build COBie mapping
cobie_mapping = {}
for config in items_to_process:
    ifc_type = config['ifc_type']
    search_term = config['excel_search']
    display_name = config['display_name']
    
    # Find rows in Excel where Title contains search term (case-insensitive)
    matched_rows = ef_df[ef_df['Title'].astype(str).str.contains(search_term, case=False, na=False)].copy()
    matched_rows = matched_rows.sort_values(by=['Title', 'Code'], kind='stable').reset_index(drop=True)
    
    if not matched_rows.empty:
        cobie_value = str(matched_rows.iloc[0]['COBie']).strip()
        cobie_mapping[ifc_type] = {
            'value': cobie_value,
            'excel_title_df': matched_rows,
            'count': len(matched_rows),
            'display_name': display_name
        }
        print(f'{display_name:10s} - COBie value found: {cobie_value}')
    else:
        print(f'{display_name:10s} - WARNING: No COBie mapping found in Excel!')

print(f'\nTotal items to process: {len(cobie_mapping)}')


Walls      - COBie value found: EF_25_10_25 : External walls
Doors      - COBie value found: EF_25_30_25 : Doors
Windows    - COBie value found: EF_25_30_97 : Windows

Total items to process: 3


In [4]:
# ============================================================
# BEFORE CHECK: Extract items from source data (before processing)
# ============================================================

before_check = {}
for ifc_type, mapping in cobie_mapping.items():
    df = extract_items_by_type(source_data, ifc_type)
    before_check[ifc_type] = {
        'count': len(df),
        'dataframe': df,
        'display_name': mapping['display_name']
    }

# Summary table
before_summary_rows = []
for ifc_type, data in before_check.items():
    before_summary_rows.append({
        'Item Type': data['display_name'],
        'IFC Type': ifc_type,
        'Count in Source': data['count'],
        'COBie Value': cobie_mapping[ifc_type]['value'],
        'Excel Rows': cobie_mapping[ifc_type]['count']
    })

before_summary_df = pd.DataFrame(before_summary_rows)

print('=' * 80)
print('BEFORE PROCESSING CHECK')
print('=' * 80)
display(before_summary_df)

print('\nDetailed item lists:')
for ifc_type, data in before_check.items():
    print(f"\n{data['display_name']} ({ifc_type}):")
    if not data['dataframe'].empty:
        display(data['dataframe'][['Count', 'GUID', 'Name', 'DbId', 'Item.Type']])
    else:
        print(f"  No items found")


BEFORE PROCESSING CHECK


,Item Type,IFC Type,Count in Source,COBie Value,Excel Rows
0,Walls,IFCWALL,1,EF_25_10_25 : External walls,8
1,Doors,IFCDOOR,14,EF_25_30_25 : Doors,1
2,Windows,IFCWINDOW,24,EF_25_30_97 : Windows,1



Detailed item lists:

Walls (IFCWALL):


,Count,GUID,Name,DbId,Item.Type
0,1,0/0/0/1/10,Basic Wall:Party Wall - CMU Residential Unit D...,684,IFCWALL



Doors (IFCDOOR):


,Count,GUID,Name,DbId,Item.Type
0,1,0/0/0/0/32,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:150173,41,IFCDOOR
1,2,0/0/0/0/33,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:150257,42,IFCDOOR
2,3,0/0/0/1/68,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:203720,742,IFCDOOR
3,4,0/0/0/1/70,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:204034,744,IFCDOOR
4,5,0/0/0/1/33,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:150378,707,IFCDOOR
5,6,0/0/0/1/34,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:150478,708,IFCDOOR
6,7,0/0/0/1/35,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:159734,709,IFCDOOR
7,8,0/0/0/1/36,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:159834,710,IFCDOOR
8,9,0/0/0/1/37,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:160065,711,IFCDOOR
9,10,0/0/0/1/38,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:160208,712,IFCDOOR



Windows (IFCWINDOW):


,Count,GUID,Name,DbId,Item.Type
0,1,0/0/0/1/27,M_Casement:819mm x 759mm:819mm x 759mm:148607,701,IFCWINDOW
1,2,0/0/0/1/31,M_Casement:819mm x 759mm:819mm x 759mm:149736,705,IFCWINDOW
2,3,0/0/0/1/50,M_Casement:819mm x 759mm:819mm x 759mm:180994,724,IFCWINDOW
3,4,0/0/0/1/53,M_Casement:819mm x 759mm:819mm x 759mm:181548,727,IFCWINDOW
4,5,0/0/0/1/25,M_Fixed:2800mm x 2410mm:2800mm x 2410mm:147686,699,IFCWINDOW
5,6,0/0/0/1/29,M_Fixed:2800mm x 2410mm:2800mm x 2410mm:149278,703,IFCWINDOW
6,7,0/0/0/1/47,M_Fixed:2800mm x 2410mm:2800mm x 2410mm:180318,721,IFCWINDOW
7,8,0/0/0/1/51,M_Fixed:2800mm x 2410mm:2800mm x 2410mm:181096,725,IFCWINDOW
8,9,0/0/0/0/26,M_Fixed:4835mm x 2420mm:4835mm x 2420mm:145788,35,IFCWINDOW
9,10,0/0/0/0/27,M_Fixed:4835mm x 2420mm:4835mm x 2420mm:146016,36,IFCWINDOW


In [5]:
# ============================================================
# APPLY COBIE VALUES (Consolidated Processing)
# ============================================================

# Build GUID lookup for working data
working_by_guid = {}
for item in working_data:
    if not isinstance(item, dict):
        continue
    guid = str(item.get('ExternalId', '')).strip()
    if guid:
        working_by_guid[guid] = item

# Process all item types and collect results
process_results = {}

for ifc_type, mapping in cobie_mapping.items():
    display_name = mapping['display_name']
    cobie_value = mapping['value']
    source_df = before_check[ifc_type]['dataframe']
    
    if source_df.empty:
        print(f'\nSkipping {display_name}: No items found in source data')
        continue
    
    print(f'\nProcessing {display_name} ({ifc_type})...')
    
    updated_count = 0
    added_count = 0
    missing_guid_count = 0
    before_values = []
    after_values = []
    
    for row in source_df.itertuples(index=False):
        target_item = working_by_guid.get(str(row.GUID).strip())
        if not isinstance(target_item, dict):
            missing_guid_count += 1
            before_values.append('(not found)')
            after_values.append('(not found)')
            continue
        
        was_updated, before_val, after_val = apply_cobie_value(target_item, cobie_value)
        
        before_values.append(before_val)
        after_values.append(after_val)
        
        if was_updated:
            updated_count += 1
            if before_val == '(none)':
                added_count += 1
    
    # Store results for after check
    process_results[ifc_type] = {
        'updated_count': updated_count,
        'added_count': added_count,
        'missing_guid_count': missing_guid_count,
        'total_items': len(source_df),
        'before_values': before_values,
        'after_values': after_values,
        'display_name': display_name
    }
    
    print(f'  Updated: {updated_count} | Added: {added_count} | Missing GUIDs: {missing_guid_count}')

# Write updated JSON
with working_json_path.open('w', encoding='utf-8') as f:
    json.dump(working_data, f, ensure_ascii=False, indent=2)

print(f'\n\n✓ Updated JSON written to: {working_json_path}')



Processing Walls (IFCWALL)...
  Updated: 0 | Added: 0 | Missing GUIDs: 0

Processing Doors (IFCDOOR)...
  Updated: 0 | Added: 0 | Missing GUIDs: 0

Processing Windows (IFCWINDOW)...
  Updated: 0 | Added: 0 | Missing GUIDs: 0


✓ Updated JSON written to: C:\Git\APS-IFC\JSON_Edit\Ifc2x3_Duplex_Architecture.json


In [6]:
# ============================================================
# AFTER CHECK: Verify results and show COBie data written to IFC
# ============================================================

print('=' * 80)
print('AFTER PROCESSING CHECK & SUMMARY')
print('=' * 80)

# Summary table of changes
after_summary_rows = []
total_updated = 0
total_added = 0
total_missing = 0

for ifc_type, results in process_results.items():
    after_summary_rows.append({
        'Item Type': results['display_name'],
        'IFC Type': ifc_type,
        'Total Items': results['total_items'],
        'Updated': results['updated_count'],
        'Added': results['added_count'],
        'Missing GUIDs': results['missing_guid_count'],
        'COBie Value': cobie_mapping[ifc_type]['value']
    })
    total_updated += results['updated_count']
    total_added += results['added_count']
    total_missing += results['missing_guid_count']

after_summary_df = pd.DataFrame(after_summary_rows)
display(after_summary_df)

# Overall statistics
print(f'\n{"OVERALL STATISTICS":^80}')
print(f'Total Items Processed: {sum(r["total_items"] for r in process_results.values())}')
print(f'Total Updated:         {total_updated}')
print(f'Total Added:           {total_added}')
print(f'Total Missing GUIDs:   {total_missing}')
print(f'Source JSON Records:   {len(source_data)}')
print(f'\nOutput Path: {working_json_path}')

# Detailed COBie values written to IFC for each item type
print(f'\n{"DETAILED COBIE DATA TO IFC":^80}')

for ifc_type, results in process_results.items():
    display_name = results['display_name']
    source_df = before_check[ifc_type]['dataframe']
    
    if source_df.empty:
        continue
    
    # Create detailed output table
    detail_df = source_df[['Count', 'GUID', 'Name', 'DbId', 'Item.Type']].copy()
    detail_df['COBie data to IFC'] = results['after_values']
    
    print(f'\n{display_name} ({ifc_type}) - COBie data written to IFC:')
    display(detail_df)
    
    # Show unique written values
    written_unique = sorted(set(results['after_values']))
    print(f'  Unique COBie data written to IFC: {written_unique}')

print(f'\n{"PROCESSING COMPLETE":^80}')
print(f'✓ All COBie values successfully applied and saved to: {working_json_path}')


AFTER PROCESSING CHECK & SUMMARY


,Item Type,IFC Type,Total Items,Updated,Added,Missing GUIDs,COBie Value
0,Walls,IFCWALL,1,0,0,0,EF_25_10_25 : External walls
1,Doors,IFCDOOR,14,0,0,0,EF_25_30_25 : Doors
2,Windows,IFCWINDOW,24,0,0,0,EF_25_30_97 : Windows



                               OVERALL STATISTICS                               
Total Items Processed: 39
Total Updated:         0
Total Added:           0
Total Missing GUIDs:   0
Source JSON Records:   1432

Output Path: C:\Git\APS-IFC\JSON_Edit\Ifc2x3_Duplex_Architecture.json

                           DETAILED COBIE DATA TO IFC                           

Walls (IFCWALL) - COBie data written to IFC:


,Count,GUID,Name,DbId,Item.Type,COBie data to IFC
0,1,0/0/0/1/10,Basic Wall:Party Wall - CMU Residential Unit D...,684,IFCWALL,EF_25_10_25 : External walls


  Unique COBie data written to IFC: ['EF_25_10_25 : External walls']

Doors (IFCDOOR) - COBie data written to IFC:


,Count,GUID,Name,DbId,Item.Type,COBie data to IFC
0,1,0/0/0/0/32,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:150173,41,IFCDOOR,EF_25_30_25 : Doors
1,2,0/0/0/0/33,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:150257,42,IFCDOOR,EF_25_30_25 : Doors
2,3,0/0/0/1/68,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:203720,742,IFCDOOR,EF_25_30_25 : Doors
3,4,0/0/0/1/70,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:204034,744,IFCDOOR,EF_25_30_25 : Doors
4,5,0/0/0/1/33,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:150378,707,IFCDOOR,EF_25_30_25 : Doors
5,6,0/0/0/1/34,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:150478,708,IFCDOOR,EF_25_30_25 : Doors
6,7,0/0/0/1/35,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:159734,709,IFCDOOR,EF_25_30_25 : Doors
7,8,0/0/0/1/36,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:159834,710,IFCDOOR,EF_25_30_25 : Doors
8,9,0/0/0/1/37,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:160065,711,IFCDOOR,EF_25_30_25 : Doors
9,10,0/0/0/1/38,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:160208,712,IFCDOOR,EF_25_30_25 : Doors


  Unique COBie data written to IFC: ['EF_25_30_25 : Doors']

Windows (IFCWINDOW) - COBie data written to IFC:


,Count,GUID,Name,DbId,Item.Type,COBie data to IFC
0,1,0/0/0/1/27,M_Casement:819mm x 759mm:819mm x 759mm:148607,701,IFCWINDOW,EF_25_30_97 : Windows
1,2,0/0/0/1/31,M_Casement:819mm x 759mm:819mm x 759mm:149736,705,IFCWINDOW,EF_25_30_97 : Windows
2,3,0/0/0/1/50,M_Casement:819mm x 759mm:819mm x 759mm:180994,724,IFCWINDOW,EF_25_30_97 : Windows
3,4,0/0/0/1/53,M_Casement:819mm x 759mm:819mm x 759mm:181548,727,IFCWINDOW,EF_25_30_97 : Windows
4,5,0/0/0/1/25,M_Fixed:2800mm x 2410mm:2800mm x 2410mm:147686,699,IFCWINDOW,EF_25_30_97 : Windows
5,6,0/0/0/1/29,M_Fixed:2800mm x 2410mm:2800mm x 2410mm:149278,703,IFCWINDOW,EF_25_30_97 : Windows
6,7,0/0/0/1/47,M_Fixed:2800mm x 2410mm:2800mm x 2410mm:180318,721,IFCWINDOW,EF_25_30_97 : Windows
7,8,0/0/0/1/51,M_Fixed:2800mm x 2410mm:2800mm x 2410mm:181096,725,IFCWINDOW,EF_25_30_97 : Windows
8,9,0/0/0/0/26,M_Fixed:4835mm x 2420mm:4835mm x 2420mm:145788,35,IFCWINDOW,EF_25_30_97 : Windows
9,10,0/0/0/0/27,M_Fixed:4835mm x 2420mm:4835mm x 2420mm:146016,36,IFCWINDOW,EF_25_30_97 : Windows


  Unique COBie data written to IFC: ['EF_25_30_97 : Windows']

                              PROCESSING COMPLETE                               
✓ All COBie values successfully applied and saved to: C:\Git\APS-IFC\JSON_Edit\Ifc2x3_Duplex_Architecture.json
